# Create CSV files
For OneDrive Folder with raw data

In [6]:
#install
#!pip install duckdb pyarrow -q

#imports
import json, os, random, duckdb
import pandas as pd
import logging
import numpy as np

os.makedirs('logs', exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/pipeline.log'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('pipeline')
log.info("Pipeline initialized.")

2026-03-31 09:52:12,419 - INFO - Pipeline initialized.


In [2]:
#load in json files
#source: https://github.com/fcakyon/instafake-dataset

try: 
    with open("data/fakeAccountData.json") as f:
        fake_data = json.load(f)
    for r in fake_data:
        r['isFake'] = 1
        r['label'] = 'fake'
    log.info(f"Loaded fakeAccountData.json: {len(fake_data)} fake accounts.")

    with open("data/realAccountData.json") as f:
        real_data = json.load(f)
    for r in real_data:
        r['isFake'] = 0
        r['label'] = 'real'
    log.info(f"Loaded realAccountData.json: {len(real_data)} real accounts.")
    df_raw = pd.DataFrame(fake_data + real_data).reset_index(drop=True)
    df_raw['account_id'] = df_raw.index + 1
    log.info(f"Combined dataset: {len(df_raw)} total accounts.")
    log.info(f"Columns in dataset: {df_raw.columns.tolist()}")
except fileNotFoundError as e:
    log.error(f"File not found: {e.filename}")
    raise
except json.JSONDecodeError as e:
    log.error(f"Error decoding JSON: {e.msg} at line {e.lineno} column {e.colno}")
    raise
except Exception as e:
    log.error(f"Unexpected error: {str(e)}")
    raise
df_raw.head()

2026-03-31 09:49:00,038 - INFO - Loaded fakeAccountData.json: 200 fake accounts.
2026-03-31 09:49:00,102 - INFO - Loaded realAccountData.json: 994 real accounts.
2026-03-31 09:49:00,115 - INFO - Combined dataset: 1194 total accounts.
2026-03-31 09:49:00,117 - INFO - Columns in dataset: ['userFollowerCount', 'userFollowingCount', 'userBiographyLength', 'userMediaCount', 'userHasProfilPic', 'userIsPrivate', 'usernameDigitCount', 'usernameLength', 'isFake', 'label', 'account_id']


,userFollowerCount,userFollowingCount,userBiographyLength,userMediaCount,userHasProfilPic,userIsPrivate,usernameDigitCount,usernameLength,isFake,label,account_id
0,25,1937,0,0,1,1,0,10,1,fake,1
1,324,4122,0,0,1,0,4,15,1,fake,2
2,15,399,0,0,0,0,3,12,1,fake,3
3,14,107,0,1,1,0,1,10,1,fake,4
4,264,4651,0,0,1,0,0,14,1,fake,5


In [3]:
#build face_images table
#source: https://www.kaggle.com/datasets/kaustubhdhote/human-faces-dataset
#images stored at data/Human Faces Dataset/

real_path = "data/Human Faces Dataset/Real Images"
fake_path = "data/Human Faces Dataset/AI-Generated Images"

try:
    if not os.path.exists(real_path):
        raise FileNotFoundError(f"Directory not found: {real_path}")
    if not os.path.exists(fake_path):
        raise FileNotFoundError(f"Directory not found: {fake_path}")
    #scan directories for image files
    real_files = sorted([f for f in os.listdir(real_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    fake_files = sorted([f for f in os.listdir(fake_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    #log images found
    log.info(f"Found {len(real_files)} real images and {len(fake_files)} fake images.")

    face_images = pd.DataFrame({
        'image_id': range(1, len(real_files) + len(fake_files) + 1),
        'filename': real_files + fake_files,
        'face_type': ['real'] * len(real_files) + ['ai'] * len(fake_files)
    })
    log.info(f"Constructed face_images DataFrame with {len(face_images)} records.")
except FileNotFoundError as e:
    log.error(f"Image folder is missing: {e}")
    raise
except Exception as e:
    log.error(f"Error processing images: {str(e)}")
    raise
face_images.head()


2026-03-31 09:49:00,378 - INFO - Found 5000 real images and 4630 fake images.
2026-03-31 09:49:00,387 - INFO - Constructed face_images DataFrame with 9630 records.


,image_id,filename,face_type
0,1,000001.jpg,real
1,2,000002.jpg,real
2,3,000003.jpg,real
3,4,000004.jpg,real
4,5,000005.jpg,real


In [4]:
#assigning a face image to each account
#real accounts get real images, fake accounts get ai images

try:
    random.seed(42)  #for reproducibility
    read_ids = face_images[face_images['face_type'] == 'real']['image_id'].tolist()
    ai_ids = face_images[face_images['face_type'] == 'ai']['image_id'].tolist()
    df_raw['profile_pic_id'] = df_raw['label'].apply(
        lambda x : random.choice(read_ids) if x == 'real' else random.choice(ai_ids)
    )

    log.info("Assigned profile picture id to accounts.")
    log.info(f"Sample of assigned profile_pic_id:\n{df_raw[['account_id', 'label', 'profile_pic_id']].head()}")
except Exception as e:
    log.error(f"Error assigning profile pictures: {str(e)}")
    raise

2026-03-31 09:49:00,446 - INFO - Assigned profile picture id to accounts.


2026-03-31 09:49:00,457 - INFO - Sample of assigned profile_pic_id:
   account_id label  profile_pic_id
0           1  fake            5913
1           2  fake            5205
2           3  fake            7254
3           4  fake            7007
4           5  fake            6829


In [19]:
#generate synthetic data for accounts without profile pics
try:
    n_real_accounts = len(df_raw)
    n_total_images = len(face_images)
    n_synthetic = n_total_images - n_real_accounts
    log.info(f"Generating {n_synthetic} synthetic accounts to match total images.")

    #learning distributions from real data
    #numeric columns will use normal distributions, categorical columns will use empirical probability

    #sample n values from a normal fit to col, clipped to a valid range
    def sample_col(col, n, clip_min=0, clip_max=None):
        mu, sigma = df_raw[col].mean(), df_raw[col].std()
        vals = np.random.normal(mu, sigma, n).round().astype(int)
        vals = np.clip(vals, clip_min, clip_max if clip_max else vals.max())
        return vals
    np.random.seed(42)
    synthetic_labels = np.random.choice(['real', 'fake'], size=n_synthetic, p=[0.5,0.5])

    df_synthetic = pd.DataFrame({
        'account_id': range(n_real_accounts + 1,
                            n_real_accounts + n_synthetic + 1),
        'label': synthetic_labels,
        'isFake': (synthetic_labels == 'fake').astype(int),
        'is_synthetic': True,

        #stats
        'userFollowerCount': sample_col('userFollowerCount',   n_synthetic,
                                           clip_min=0, clip_max=4492),
        'userFollowingCount': sample_col('userFollowingCount',  n_synthetic,
                                           clip_min=0, clip_max=7497),
        'userMediaCount': sample_col('userMediaCount',      n_synthetic,
                                           clip_min=0, clip_max=1058),

        #profile
        'userBiographyLength': sample_col('userBiographyLength', n_synthetic,
                                           clip_min=0, clip_max=150),
        'userHasProfilPic': np.random.binomial(
                                   1, df_raw['userHasProfilPic'].mean(),
                                   n_synthetic),
        'userIsPrivate': np.random.binomial(
                                   1, df_raw['userIsPrivate'].mean(),
                                   n_synthetic),
        'usernameDigitCount': sample_col('usernameDigitCount',  n_synthetic,
                                           clip_min=0, clip_max=10),
        'usernameLength': sample_col('usernameLength',      n_synthetic,
                                           clip_min=5, clip_max=30),
    })

    log.info(f"Synthetic data generated: {df_synthetic.shape}")
    log.info(f"Synthetic label split:\n{df_synthetic['label'].value_counts().to_string()}")

except Exception as e:
    log.error(f"Error generating synthetic accounts: {e}")
    raise

2026-03-31 10:51:13,771 - INFO - Generating 8436 synthetic accounts to match total images.


2026-03-31 10:51:13,977 - INFO - Synthetic data generated: (8436, 12)
2026-03-31 10:51:14,018 - INFO - Synthetic label split:
label
real    4278
fake    4158


In [20]:
#combine real and synthetic dataframes and assign the remaining images

try:
    df_raw['is_synthetic'] = False

    #combined
    df_all = pd.concat([df_raw, df_synthetic], ignore_index=True)
    df_all['account_id'] = df_all.index + 1  #reassign account_id to be unique and sequential

    log.info(f"Combined real and synthetic data: {df_all.shape}")
    log.info(f"Final label distribution:\n{df_all['label'].value_counts().to_string()}")

    #assign remaining images
    face_images_shuffled = face_images.copy()

    real_img_ids = face_images_shuffled[face_images_shuffled['face_type'] == 'real']['image_id'].tolist()
    ai_img_ids = face_images_shuffled[face_images_shuffled['face_type'] == 'ai']['image_id'].tolist()

    random.seed(42)
    random.shuffle(real_img_ids)
    random.shuffle(ai_img_ids)

    real_iter = iter(real_img_ids)
    ai_iter = iter(ai_img_ids)

    #assign real images to real accounts, ai images to fake accounts randomly
    #switched from filename to profile_pic_id
    def assign_unique(label):
        try:
            return next(real_iter) if label == 'real' else next(ai_iter)
        except StopIteration:
            return random.choice(real_img_ids if label == 'real' else ai_img_ids)

    df_all['profile_pic_id'] = df_all['label'].apply(assign_unique)

    log.info("Assigned profile pictures to all accounts.")
    log.info(f"Sample of final dataset:\n{df_all[['account_id', 'label', 'is_synthetic', 'profile_pic_id']].head()}")

except Exception as e:
    log.error(f"Error combining datasets and assigning images: {e}")
    raise

2026-03-31 10:51:16,629 - INFO - Combined real and synthetic data: (9630, 13)
2026-03-31 10:51:16,633 - INFO - Final label distribution:
label
real    5272
fake    4358


2026-03-31 10:51:16,725 - INFO - Assigned profile pictures to all accounts.
2026-03-31 10:51:16,742 - INFO - Sample of final dataset:
   account_id label  is_synthetic  profile_pic_id
0           1  fake         False            7409
1           2  fake         False            5840
2           3  fake         False            8423
3           4  fake         False            5191
4           5  fake         False            6494


In [22]:
# normalize into 4 relational tables
#accounts, accounts_stats, account_profile, face_images (made previously)

try: 
    accounts = df_all[['account_id', 'label', 'isFake', 'profile_pic_id']].copy()

    account_stats = df_all[['account_id',
                             'userFollowerCount',
                             'userFollowingCount',
                             'userMediaCount']].copy()
    account_stats.columns = ['account_id', 'follower_count',
                              'following_count', 'media_count']

    account_profile = df_all[['account_id',
                               'userBiographyLength',
                               'userHasProfilPic',
                               'userIsPrivate',
                               'usernameDigitCount',
                               'usernameLength']].copy()
    account_profile.columns = ['account_id', 'bio_length', 'has_profile_pic',
                                'is_private', 'username_digit_count', 'username_length']

    for name, tbl in [('accounts', accounts),
                      ('account_stats', account_stats),
                      ('account_profile', account_profile),
                      ('face_images', face_images)]:
        log.info(f"Table '{name}': {tbl.shape[0]} rows x {tbl.shape[1]} cols")

except KeyError as e:
    log.error(f"Column not found, check if JSON field names match: {e}")
    raise
except Exception as e:
    log.error(f"Error normalizing tables: {e}")
    raise

2026-03-31 10:51:58,850 - INFO - Table 'accounts': 9630 rows x 4 cols
2026-03-31 10:51:58,852 - INFO - Table 'account_stats': 9630 rows x 4 cols


2026-03-31 10:51:58,853 - INFO - Table 'account_profile': 9630 rows x 6 cols
2026-03-31 10:51:58,856 - INFO - Table 'face_images': 9630 rows x 3 cols


In [23]:
#export as CSV

try:
    os.makedirs("data/csv/", exist_ok=True)
    accounts.to_csv("data/csv/accounts.csv", index=False)
    account_stats.to_csv("data/csv/account_stats.csv", index=False)
    account_profile.to_csv("data/csv/account_profile.csv", index=False)
    face_images.to_csv("data/csv/face_images.csv", index=False)

    total = sum(os.path.getsize(f"data/csv/{fname}") for fname in os.listdir("data/csv/") if fname.endswith('.csv'))

    log.info(f"CSVs written to 'data/csv/' with total size {total / 1024:.1f} KB")

except PermissionError as e:
    log.error(f"Permission error writing CSV: {e}")
    raise
except Exception as e:
    log.error(f"Error exporting CSVs: {e}")
    raise

2026-03-31 10:52:03,154 - INFO - CSVs written to 'data/csv/' with total size 643.3 KB


In [24]:
#export as parquet

try:
    os.makedirs("data/parquet/", exist_ok=True)
    accounts.to_parquet("data/parquet/accounts.parquet", index=False)
    account_stats.to_parquet("data/parquet/account_stats.parquet", index=False)
    account_profile.to_parquet("data/parquet/account_profile.parquet", index=False)
    face_images.to_parquet("data/parquet/face_images.parquet", index=False)

    total = sum(os.path.getsize(f"data/parquet/{fname}") for fname in os.listdir("data/parquet/") if fname.endswith('.parquet'))

    log.info(f"Parquets written to 'data/parquet/' with total size {total / 1024:.1f} KB")
except Exception as e:
    log.error(f"Error exporting Parquets: {e}")
    raise

2026-03-31 10:52:08,712 - INFO - Parquets written to 'data/parquet/' with total size 400.7 KB


# Data Preparation

Demonstrate how to load your data files into a database with DuckDB using python

In [ ]:
#load into duckdb and turn tables into duckdb tables for querying

try:
    con = duckdb.connect("data/fake_detection.duckdb")
    log.info("Connected to DuckDB.")

    tables = {
        'accounts': "data/parquet/accounts.parquet",
        'account_stats': "data/parquet/account_stats.parquet",
        'account_profile': "data/parquet/account_profile.parquet",
        'face_images': "data/parquet/face_images.parquet"
    }
    for name, path in tables.items():
        con.execute(f"CREATE TABLE {name} AS SELECT * FROM read_parquet('{path}')")
        n = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        log.info(f"Loaded table '{name}' from {path} with {n} rows")

except duckdb.Error as e:
    log.error(f"DuckDB error: {e}")
    raise
except Exception as e:
    log.error(f"Error loading data into DuckDB: {e}")
    raise

2026-03-31 10:37:56,108 - INFO - Connected to DuckDB.
2026-03-31 10:37:56,219 - INFO - Loaded table 'accounts' from data/parquet/accounts.parquet with 9630 rows
2026-03-31 10:37:56,254 - INFO - Loaded table 'account_stats' from data/parquet/account_stats.parquet with 9630 rows
2026-03-31 10:37:56,301 - INFO - Loaded table 'account_profile' from data/parquet/account_profile.parquet with 9630 rows
2026-03-31 10:37:56,340 - INFO - Loaded table 'face_images' from data/parquet/face_images.parquet with 9630 rows


In [28]:
#join all tables for a model-ready flat table

try: 
    df_model_ready = con.execute("""
        SELECT
            a.account_id,
            a.label,
            a.isFake,
            p.bio_length,
            p.has_profile_pic,
            p.is_private,
            p.username_digit_count,
            p.username_length,
            s.follower_count,
            s.following_count,
            s.media_count,
            f.image_id AS profile_pic_id,
            f.face_type AS profile_pic_type,
            f.filename AS profile_pic_filename
        FROM accounts a
            JOIN account_stats s ON a.account_id = s.account_id
            JOIN account_profile p ON a.account_id = p.account_id
            JOIN face_images f ON a.profile_pic_id = f.image_id""").df()
    log.info(f"Joined tables into model-ready DataFrame with {df_model_ready.shape[0]} rows and {df_model_ready.shape[1]} columns.")
except duckdb.Error as e:
    log.error(f"DuckDB query error: {e}")
    raise
except Exception as e:
    log.error(f"Error joining tables: {e}")
    raise

df_model_ready.head()                                

2026-03-31 10:57:58,699 - INFO - Joined tables into model-ready DataFrame with 9630 rows and 14 columns.


,account_id,label,isFake,bio_length,has_profile_pic,is_private,username_digit_count,username_length,follower_count,following_count,media_count,profile_pic_id,profile_pic_type,profile_pic_filename
0,407,real,0,0,1,1,0,9,253,303,7,1,real,000001.jpg
1,9055,real,0,7,1,0,0,11,587,0,103,2,real,000002.jpg
2,6120,real,0,0,1,1,0,14,967,0,0,3,real,000003.jpg
3,5427,real,0,21,1,1,2,12,853,991,41,4,real,000004.jpg
4,7608,real,0,0,1,0,0,14,958,571,46,5,real,000005.jpg


# Solution analysis

Implement a model 

In [29]:
#install tensorflow for modeling
%pip install tensorflow -q

Note: you may need to restart the kernel to use updated packages.


In [31]:
#imports
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

log.info(f"TensorFlow version: {tf.__version__}")

I0000 00:00:1774970751.044052   10759 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774970757.463537   10759 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774970783.061368   10759 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-31 11:26:38,655 - INFO - TensorFlow version: 2.21.0


In [36]:
#build image dataset from df_model_ready

try:
    #build dataframe with full path + binary label
    def get_full_path(row):
        if row['profile_pic_type'] == 'real':
            return os.path.join(real_path, row['profile_pic_filename'])
        else:
            return os.path.join(fake_path, row['profile_pic_filename'])
        
    df_images = df_model_ready[['profile_pic_filename', 'profile_pic_type']].copy()
    df_images['full_path'] = df_images.apply(get_full_path, axis=1)
    df_images['label_binary'] = (df_images['profile_pic_type'] == 'ai').astype(int)

    #verify files exist
    missing = df_images[~df_images['full_path'].apply(os.path.exists)]
    if len(missing) > 0:
        log.warning(f"{len(missing)} image files not found on disk")
    else:
        log.info(f"All {len(df_images)} image files verified on disk")

    log.info(f"Image dataset: {df_images['profile_pic_type'].value_counts().to_string()}")

except Exception as e:
    log.error(f"Error building image dataset: {e}")
    raise

df_images.head()

2026-03-31 11:32:18,906 - INFO - All 9630 image files verified on disk
2026-03-31 11:32:18,932 - INFO - Image dataset: profile_pic_type
real    8065
ai      1565


,profile_pic_filename,profile_pic_type,full_path,label_binary
0,000001.jpg,real,data/Human Faces Dataset/Real Images/000001.jpg,0
1,000002.jpg,real,data/Human Faces Dataset/Real Images/000002.jpg,0
2,000003.jpg,real,data/Human Faces Dataset/Real Images/000003.jpg,0
3,000004.jpg,real,data/Human Faces Dataset/Real Images/000004.jpg,0
4,000005.jpg,real,data/Human Faces Dataset/Real Images/000005.jpg,0


In [37]:
#train and validate split for images
from sklearn.model_selection import train_test_split

try:
    train_df, val_df = train_test_split(
        df_images,
        test_size=0.2,
        random_state=42,
        stratify=df_images['profile_pic_type']
    )

    log.info(f"Train set: {len(train_df)} images")
    log.info(f"Val set:   {len(val_df)} images")

except Exception as e:
    log.error(f"Error splitting image data: {e}")
    raise

2026-03-31 11:34:00,774 - INFO - Train set: 7704 images
2026-03-31 11:34:00,790 - INFO - Val set:   1926 images


In [38]:
#image data generators 
#MobileNetV2 expects 224x224 images, so we will resize and rescale pixel values
IMG_SIZE  = (224, 224)
BATCH     = 32

train_gen = ImageDataGenerator(
    rescale=1./127.5,
    preprocessing_function=lambda x: x - 1,  # scale to [-1, 1]
    horizontal_flip=True,
    rotation_range=10,
    zoom_range=0.1
)

val_gen = ImageDataGenerator(
    rescale=1./127.5,
    preprocessing_function=lambda x: x - 1
)

train_flow = train_gen.flow_from_dataframe(
    train_df,
    x_col='full_path',
    y_col='profile_pic_type',
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='binary',
    classes=['real', 'ai'],   # real=0, ai=1
    shuffle=True,
    seed=42
)

val_flow = val_gen.flow_from_dataframe(
    val_df,
    x_col='full_path',
    y_col='profile_pic_type',
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='binary',
    classes=['real', 'ai'],
    shuffle=False
)

log.info(f"Train batches: {len(train_flow)}, Val batches: {len(val_flow)}")


Found 7704 validated image filenames belonging to 2 classes.
Found 1926 validated image filenames belonging to 2 classes.


2026-03-31 11:35:57,112 - INFO - Train batches: 241, Val batches: 61


In [39]:
#build a MobileNetV2 model
try:
    base_model = MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,       #remove ImageNet classification head
        weights='imagenet'       #use pretrained weights
    )

    #freeze base layers, so we only train our custom head
    base_model.trainable = False

    #add custom classification head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)   #binary: real vs ai

    cnn_model = Model(inputs=base_model.input, outputs=output)

    cnn_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    log.info(f"MobileNetV2 model built")
    log.info(f"Trainable params: {sum([tf.size(w).numpy() for w in cnn_model.trainable_weights]):,}")

except Exception as e:
    log.error(f"Error building CNN model: {e}")
    raise

E0000 00:00:1774971416.405437   10759 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


2026-03-31 11:37:03,875 - INFO - MobileNetV2 model built
2026-03-31 11:37:03,930 - INFO - Trainable params: 164,097


In [40]:
#train the cnn

#train only the custom head with frozen base layers 
try:
    history = cnn_model.fit(
        train_flow,
        epochs=5,
        validation_data=val_flow,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                patience=2,
                restore_best_weights=True
            )
        ]
    )

    os.makedirs("models", exist_ok=True)
    cnn_model.save("models/mobilenet_facedetector.h5")
    log.info("CNN training complete — model saved to models/mobilenet_facedetector.h5")

except Exception as e:
    log.error(f"Error training CNN: {e}")
    raise

Epoch 1/5


I0000 00:00:1774971465.043408   10759 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


241/241 ━━━━━━━━━━━━━━━━━━━━ 436s 2s/step - accuracy: 0.9585 - loss: 0.1202 - val_accuracy: 0.9943 - val_loss: 0.0270
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 272s 1s/step - accuracy: 0.9926 - loss: 0.0264 - val_accuracy: 0.9943 - val_loss: 0.0180
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 221s 915ms/step - accuracy: 0.9949 - loss: 0.0173 - val_accuracy: 0.9969 - val_loss: 0.0137
Epoch 4/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accuracy: 0.9966 - loss: 0.0111 - val_accuracy: 0.9990 - val_loss: 0.0074
Epoch 5/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 275s 1s/step - accuracy: 0.9971 - loss: 0.0099 - val_accuracy: 0.9979 - val_loss: 0.0095


2026-03-31 12:02:42,643 - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2026-03-31 12:02:43,861 - INFO - CNN training complete — model saved to models/mobilenet_facedetector.h5


In [41]:
#evaluate cnn
from sklearn.metrics import classification_report, confusion_matrix

try:
    val_preds_prob = cnn_model.predict(val_flow)
    val_preds = (val_preds_prob > 0.5).astype(int).flatten()
    val_true  = val_flow.labels

    log.info("CNN Classification Report:")
    report = classification_report(val_true, val_preds,
                                   target_names=['real', 'ai'])
    log.info(f"\n{report}")
    print(report)

    cm = confusion_matrix(val_true, val_preds)
    log.info(f"Confusion matrix:\n{cm}")

except Exception as e:
    log.error(f"Error evaluating CNN: {e}")
    raise

61/61 ━━━━━━━━━━━━━━━━━━━━ 44s 658ms/step


2026-03-31 12:03:34,534 - INFO - CNN Classification Report:
2026-03-31 12:03:34,777 - INFO - 
              precision    recall  f1-score   support

        real       1.00      1.00      1.00      1613
          ai       0.99      1.00      1.00       313

    accuracy                           1.00      1926
   macro avg       1.00      1.00      1.00      1926
weighted avg       1.00      1.00      1.00      1926

2026-03-31 12:03:34,838 - INFO - Confusion matrix:
[[1611    2]
 [   0  313]]


              precision    recall  f1-score   support

        real       1.00      1.00      1.00      1613
          ai       0.99      1.00      1.00       313

    accuracy                           1.00      1926
   macro avg       1.00      1.00      1.00      1926
weighted avg       1.00      1.00      1.00      1926



# Analysis rationale
Explain the decisions in your analysis process

# Visualization and Rationale

Create a visualization to show your results. Explain the decisions in your visualization process.